<center><h1>Yilong_Xu_HW3</h1></center>
<br>
<br>

Name: Yilong Xu
<br>
Github Username: yilongxu-usc
<br>
USC ID: 9633003204

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import bootstrap
import matplotlib.pyplot as plt
import seaborn as sns

Get the AReM Data Set

In [2]:
# Define the dataset directory
data_dir = Path("../data/AReM")

if not (data_dir / "bending1").is_dir():
    data_dir = Path(".")

# Activity folders
activities = [
    "bending1",
    "bending2",
    "cycling",
    "lying",
    "sitting",
    "standing",
    "walking"
]

# Collect dataset file paths for each activity
data_files = {}

for activity in activities:
    folder = data_dir / activity

    files = sorted(
        folder.glob("dataset*.csv"),
        key=lambda x: int(x.stem.replace("dataset", ""))
    )

    data_files[activity] = files


### (b) Test and Train Data

In [3]:
# Split datasets into training and testing sets
train_files = {}
test_files = {}

for activity, files in data_files.items():
    if activity in ["bending1", "bending2"]:
        test_files[activity] = files[:2]
        train_files[activity] = files[2:]
    else:
        test_files[activity] = files[:3]
        train_files[activity] = files[3:]

# Count training and testing instances
train_count = sum(len(files) for files in train_files.values())
test_count = sum(len(files) for files in test_files.values())

print("Training instances:", train_count)
print("Testing instances:", test_count)
print("Total instances:", train_count + test_count)

Training instances: 69
Testing instances: 19
Total instances: 88


### (c) Feature Extraction

#### i. Research

Common time-domain features used in time series classification include:
<br>
Central tendency: Mean, median, and mode.
<br>
Minimum and maximum: The smallest and largest values in a time series.
<br>
Quartiles and percentiles: First quartile (Q1), third quartile (Q3), and other percentiles.
<br>
Variability: Standard deviation, variance, range, and interquartile range (IQR).
<br>
Distribution shape: Skewness and kurtosis.
<br>
Other statistical features: Root mean square (RMS), coefficient of variation, and energy.
<br>
Temporal features: Autocorrelation, mean absolute change, number of peaks, and number of zero crossings.
<br>
These features describe the distribution, variability, and changes in time-series data. They can help identify different patterns and distinguish between different human activities.
<br>
Reference: tsfresh Documentation — Overview on Extracted Features(https://tsfresh.readthedocs.io/en/latest/text/list_of_features.html)

#### ii. Extraction

In [4]:
# Column names
columns = [
    "time", "avg_rss12", "var_rss12",
    "avg_rss13", "var_rss13",
    "avg_rss23", "var_rss23"
]

# Extract time-domain features
feature_rows = []

for activity, files in data_files.items():
    for file in files:
        df = pd.read_csv(
            file,
            comment="#",
            header=None,
            names=columns,
            sep=r"[,\s]+",
            engine="python",
            usecols=range(7)
        )

        features = {}

        for i, col in enumerate(columns[1:], start=1):
            series = df[col]

            features[f"min{i}"] = series.min()
            features[f"max{i}"] = series.max()
            features[f"mean{i}"] = series.mean()
            features[f"median{i}"] = series.median()
            features[f"std{i}"] = series.std()
            features[f"1st_quart{i}"] = series.quantile(0.25)
            features[f"3rd_quart{i}"] = series.quantile(0.75)

        feature_rows.append(features)

# Create the feature DataFrame
features_df = pd.DataFrame(feature_rows)
features_df.index = range(1, len(features_df) + 1)
features_df.index.name = "Instance"

display(features_df)
print("Feature dataset shape:", features_df.shape)

,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
Instance,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,1.30,0.358604,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,1.22,0.372437,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,1.70,0.426250,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,3.00,0.696042,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,2.83,0.535979,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,14.67,4.576562,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
85,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,13.47,4.456333,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
86,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,12.47,4.371958,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350


Feature dataset shape: (88, 42)


#### iii. Standard Deviation

In [5]:
# Estimate standard deviations and 90% bootstrap confidence intervals
bootstrap_results = []

for feature in features_df.columns:
    values = features_df[feature].dropna().to_numpy()

    # Sample standard deviation
    std_est = np.std(values, ddof=1)

    # Bootstrap 90% confidence interval
    result = bootstrap(
        (values,),
        statistic=lambda x, axis: np.std(x, axis=axis, ddof=1),
        confidence_level=0.90,
        n_resamples=10000,
        method="percentile",
        random_state=42
    )

    bootstrap_results.append({
        "Feature": feature,
        "Standard Deviation": std_est,
        "90% CI Lower": result.confidence_interval.low,
        "90% CI Upper": result.confidence_interval.high
    })

# Create results DataFrame
bootstrap_df = pd.DataFrame(bootstrap_results)
bootstrap_df = bootstrap_df.set_index("Feature")

display(bootstrap_df.round(4))
print("Results shape:", bootstrap_df.shape)

,Standard Deviation,90% CI Lower,90% CI Upper
Feature,,,
min1,9.5700,8.2672,10.7818
max1,4.3944,3.3508,5.2999
mean1,5.3357,4.7043,5.8894
median1,5.4401,4.7869,6.0092
std1,1.7722,1.5651,1.9503
1st_quart1,6.1536,5.5543,6.6536
3rd_quart1,5.1389,4.3302,5.8448
min2,0.0000,0.0000,0.0000
max2,5.0627,4.6221,5.4023


Results shape: (42, 3)


#### iv. Select Features

I would select mean, maximum, and standard deviation as the three most important time-domain features.
<br>
The mean represents the average sensor reading and can help distinguish activities with different typical values. The maximum captures the highest sensor reading during an activity, which may be useful for identifying differences in peak measurements. Standard deviation measures how much the sensor readings change over time, which could help distinguish activities with more movement, such as walking and cycling, from relatively stationary activities like sitting and lying.
<br>
I chose these three features because they describe different characteristics of the time series, including central tendency, extreme values, and variability. Also, the results from Question 1(c)iii show that some minimum features have no variation across instances, making them less useful for classification.

## 2. ISLR 3.7.4

### (a) Linear Train

I would expect the cubic regression to have a lower or equal training RSS than the linear regression. Even though the true relationship is linear, the cubic model is more flexible because it includes additional polynomial terms. These terms allow the model to fit the training data more closely, including some of the random noise. Therefore, the cubic model cannot have a higher training RSS than the linear model.

### (b) Linear Test

I would expect the linear regression to have a lower test RSS. Since the true relationship is linear, the linear model already represents the correct relationship. The cubic model has unnecessary additional terms, which may cause overfitting to the training data. As a result, the cubic model may perform worse on new data.

### (c) Not Linear Train

I would expect the cubic regression to have a lower or equal training RSS. Since the cubic model is more flexible than the linear model, it can fit the training observations at least as well. This is true even when we do not know how nonlinear the actual relationship is.

### (d) Not Linear Testing

There is not enough information to determine which model will have a lower test RSS. If the true relationship is strongly nonlinear and can be approximated well by a cubic function, the cubic regression may perform better. However, if the relationship is only slightly nonlinear, the simpler linear regression may perform better because it has lower variance and is less likely to overfit. Therefore, the result depends on the true relationship and the trade-off between bias and variance.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 